# DOE: a reward-shape sweep engineered to actually **rank** the shapes

This notebook is a **thin demo**: every step calls into the tested `geap_tuning`
package. See [`docs/doe/rlft-reward-ranking/README.md`](../docs/doe/rlft-reward-ranking/README.md).

Its sibling [`15_doe_reward_types.ipynb`](15_doe_reward_types.ipynb) swept the same
four reward shapes and got a **flat null result** — every shape *and* the untuned
baseline scored 1.000 — because the base model saturated the task and the
`Answer: <n>` marker was handed to it for free. This redesign makes the shapes
diverge while **reusing the orchestration unchanged** (`doe.run_sweep` + one
single-run `SweepConfig` per shape):

1. **Harder, tiered, larger bank** — `rlft.bench.HARD_MATH_PROBLEMS` (150 multi-step
   problems across easy/medium/hard; stratified split → test n≈30).
2. **Weaker base** — `gemini-2.5-flash-lite` for correctness headroom (step up to
   `gemini-2.5-flash` if it will not tune in your region).
3. **Neutral system instruction** — `bench.NEUTRAL_SYSTEM_INSTRUCTION` drops the
   `Answer: <n>` contract → format headroom for the `string-match` reward.
4. **Multi-objective scoring** — `run_rlft_multimetric_eval` scores `correctness`,
   `format_rate`, and `explanation_quality` (plus a per-tier breakdown), and
   `bootstrap_ci` reports each with a 95% CI so "best shape" is significance-aware.

> **Requires live GCP and incurs tuning cost** (~4 RLFT jobs). Have a real `.env`
> and `gcloud auth` in place. Charts need the viz group (`uv sync --group viz`).
> The explanation-quality judge (`gemini-2.5-pro`) is **distinct** from the
> training autorater (`gemini-2.5-flash`) to avoid grading with the trainer.

In [ ]:
from geap_tuning.config import genai_client, load_config

cfg = load_config()
client = genai_client(cfg)  # tuning is regional-only; global excludes tuning
cfg

## 1. Build the harder tiered bench and the four reward shapes

The bench is staged to GCS (train/val); the stratified test split is held out
locally. The **neutral** system instruction (no `Answer:` marker) is what gives
the format-only `string-match` reward something to teach. The autorater judge
needs a fully-qualified publisher path, reused by the `composite` blend.

In [ ]:
from geap_tuning.gcs import upload_file
from geap_tuning.rlft.bench import (
    HARD_MATH_PROBLEMS,
    NEUTRAL_SYSTEM_INSTRUCTION,
    build_bench_dataset,
    build_bench_records,
    split_stratified,
)
from geap_tuning.rlft.tune import (
    build_autorater_reward_config,
    build_composite_reward_config,
    build_reward_config,
    build_string_match_reward_config,
)

paths = build_bench_dataset("../datasets/rlft_bench", system_instruction=NEUTRAL_SYSTEM_INSTRUCTION)
train_uri = upload_file(paths["train"], f"{cfg.bucket}/doe_rlft_reward_ranking/train.jsonl")
val_uri = upload_file(paths["val"], f"{cfg.bucket}/doe_rlft_reward_ranking/val.jsonl")

train_problems, _, test_problems = split_stratified(HARD_MATH_PROBLEMS)
train_records = build_bench_records(train_problems, system_instruction=NEUTRAL_SYSTEM_INSTRUCTION)
test_records = build_bench_records(test_problems, system_instruction=NEUTRAL_SYSTEM_INSTRUCTION)

autorater_model = (
    f"projects/{cfg.project}/locations/{cfg.location}/publishers/google/models/gemini-2.5-flash"
)
reward_shapes = {
    "string-match": {"reward_config": build_string_match_reward_config()},
    "code-exec": {"reward_config": build_reward_config()},
    "autorater": {"reward_config": build_autorater_reward_config(autorater_model=autorater_model)},
    "composite": {
        "composite_reward_config": build_composite_reward_config(
            [
                (build_reward_config(), 0.8),
                (build_autorater_reward_config(autorater_model=autorater_model), 0.2),
            ]
        )
    },
}
print(f"{len(train_records)} train, {len(test_records)} test problems")
list(reward_shapes)

## 2. Pilot gate — confirm headroom *before* spending on four jobs

Score the **untuned** base on the held-out test. Proceed only if correctness is
below ceiling **and** the marker rate is low (the `Answer:` marker is no longer
free). Gemini 2.x *inference* runs on the `global` endpoint, so route the baseline
and the judge there. The judge model differs from the training autorater.

In [ ]:
import re

from geap_tuning.inference import generate
from geap_tuning.rlft.evaluate import run_rlft_multimetric_eval

BASE_MODEL = "gemini-2.5-flash-lite"  # weaker base for headroom; verify RLFT support
JUDGE_MODEL = "gemini-2.5-pro"  # distinct from the training autorater
AXES = ("correctness", "format_rate", "explanation_quality")
CORRECTNESS_CEILING, FORMAT_CEILING = 0.7, 0.5
_SCORE_RE = re.compile(r"(?:0?\.\d+|[01](?:\.0+)?)")

base_client = genai_client(cfg, base_model=BASE_MODEL)
judge_client = genai_client(cfg, base_model=JUDGE_MODEL)


def judge_fn(question: str, reply: str, truth: str) -> float:
    prompt = (
        "You are grading a math explanation. Rate how clear and correct the "
        "reasoning is on a scale from 0.0 to 1.0. Respond with ONLY the number.\n\n"
        f"Problem: {question}\nCorrect answer: {truth}\nStudent's answer: {reply}"
    )
    # gemini-2.5-pro rejects thinking_budget=0 (the generate() default); -1 = dynamic.
    match = _SCORE_RE.search(generate(judge_client, JUDGE_MODEL, prompt, thinking_budget=-1))
    return max(0.0, min(1.0, float(match.group()))) if match else 0.0


baseline = run_rlft_multimetric_eval(
    test_records,
    generate_fn=lambda u, s: generate(base_client, BASE_MODEL, u, system_instruction=s),
    judge_fn=judge_fn,
)
has_headroom = (
    baseline["correctness"] < CORRECTNESS_CEILING and baseline["format_rate"] < FORMAT_CEILING
)
print(
    f"untuned {BASE_MODEL}: correctness={baseline['correctness']:.3f} "
    f"format_rate={baseline['format_rate']:.3f} "
    f"quality={baseline.get('explanation_quality', 0.0):.3f}"
)
print(
    "Pilot gate:",
    "PASSED — headroom confirmed"
    if has_headroom
    else "FAILED — pick a weaker base / harder tier before launching",
)

## 3. Preflight every reward on one record

`validate_reward_config` scores each reward on a single example before we spend
money — RLFT auto-stops if >80% of reward calls fail. A non-null `error`/`NaN`
means the reward is broken.

In [ ]:
from geap_tuning.rlft.tune import validate_reward_config

for label, kw in reward_shapes.items():
    preflight = validate_reward_config(
        client,
        project=cfg.project,
        location=cfg.location,
        sample_answer="Answer: 4",
        example_record=train_records[0],
        reward_config=kw.get("reward_config"),
        composite_reward_config=kw.get("composite_reward_config"),
    )
    print(f"[{label}] {preflight}")

## 4. Run each reward shape as its own single-run sweep

`method="RLFT"` selects `launch_rlft_job`; each shape is a `SweepConfig` with an
**empty grid** (one run) and its reward in `fixed`. All share one Experiment and
the same multi-axis offline scorer, which replays each record's `systemInstruction`
so inference matches training. Reruns reuse finished jobs via the deterministic
display name `geap-doe-<name>-default`.

In [ ]:
from geap_tuning.config import genai_client_for_endpoint
from geap_tuning.doe import SweepConfig, run_sweep
from geap_tuning.experiments import init_experiment

EXPERIMENT_NAME = "geap-doe-rlft-reward-ranking"
init_experiment(EXPERIMENT_NAME, project=cfg.project, location=cfg.location)


def evaluate_fn(endpoint: str) -> dict:
    eval_client = genai_client_for_endpoint(cfg, endpoint)
    return run_rlft_multimetric_eval(
        test_records,
        generate_fn=lambda u, s, e=endpoint: generate(eval_client, e, u, system_instruction=s),
        judge_fn=judge_fn,
    )


results_by_run = {"untuned": baseline}
for label, kw in reward_shapes.items():
    sweep = SweepConfig(name=f"rew-rank-{label}", method="RLFT", base_model=BASE_MODEL, fixed=kw)
    result = run_sweep(
        client,
        sweep,
        train_uri=train_uri,
        val_uri=val_uri,
        evaluate_fn=evaluate_fn,
        experiment=EXPERIMENT_NAME,
        labels=cfg.labels,
    )[0]
    results_by_run[label] = result.metrics
    tag = "reused" if result.reused else "launched"
    print(
        f"  {label}: correctness={result.metrics['correctness']:.3f} "
        f"format_rate={result.metrics['format_rate']:.3f} "
        f"quality={result.metrics.get('explanation_quality', 0.0):.3f} ({tag})"
    )

## 5. Per-axis leaderboard + bootstrap confidence intervals

Rank the shapes on **each** axis (correctness is primary) and report correctness
with a bootstrap 95% CI, so the gap over the runner-up / baseline is reported
*with* whether it is significant — not an arbitrary `max`.

In [ ]:
from geap_tuning.rlft.evaluate import bootstrap_ci

rows = [
    {"run": run, **{axis: m.get(axis, 0.0) for axis in AXES}} for run, m in results_by_run.items()
]

for axis in AXES:
    ranked = sorted(rows, key=lambda r: r[axis], reverse=True)
    print(f"{axis}: winner={ranked[0]['run']} ({ranked[0][axis]:.3f})")

print("\nCorrectness with bootstrap 95% CI:")
for run, m in results_by_run.items():
    low, high = bootstrap_ci(int(m["content_hits"]), int(m["n"]))
    print(f"  {run:>14}: {m['correctness']:.3f}  CI[{low:.3f}, {high:.3f}]")

## 6. Chart the axes (needs the viz group)

Grouped bars over the three axes, then per-tier correctness (a good reward should
help most on medium/hard, where there is headroom).

In [ ]:
from geap_tuning.viz import plot_grouped_metric_bars

plot_grouped_metric_bars(rows, metrics=AXES)

In [ ]:
tier_rows = [
    {
        "run": run,
        **{
            t: m.get("by_difficulty", {}).get(t, {}).get("correctness", 0.0)
            for t in ("easy", "medium", "hard")
        },
    }
    for run, m in results_by_run.items()
]
plot_grouped_metric_bars(tier_rows, metrics=("easy", "medium", "hard"))

## 7. Read the tracked runs back from Experiments

`experiment_dataframe` returns a pandas table matching **Agent Platform Studio →
Experiments** — the four tuned shapes (the untuned baseline is offline-only).
[`11_multi_run_viz.ipynb`](11_multi_run_viz.ipynb) charts this experiment with
**zero tuning cost** (`--experiment geap-doe-rlft-reward-ranking`).

In [ ]:
from geap_tuning.experiments import experiment_dataframe

experiment_dataframe(EXPERIMENT_NAME)